In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("okienka").getOrCreate()
spark

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 34632)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/conda/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/usr/local/spark/python/pyspark/accumulators.py", line 299, in handle
    poll(accum_updates)
  File "/usr/local/spark/python/pyspark/accumulators.py", line 271, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/usr/local/spark/python/pyspark/accumulators.py", line 275, in accum_updates
    num_updates =

In [3]:
czujnik_temperatury = ((12.5, "2019-01-02 12:00:00"),
(17.6, "2019-01-02 12:00:20"),
(14.6,  "2019-01-02 12:00:30"),
(22.9,  "2019-01-02 12:01:15"),
(17.4,  "2019-01-02 12:01:30"),
(25.8,  "2019-01-02 12:03:25"),
(27.1,  "2019-01-02 12:02:40"),
)

In [5]:
# przypisujemy typy danych do kolumn
from pyspark.sql.functions import to_timestamp

from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema = StructType([
    StructField("temperatura", DoubleType(), True),
    StructField("czas", StringType(), True),
])

In [8]:
SCHEMA = """
temperatura DOUBLE, czas STRING
""" #DDL

In [10]:
# tworzenue tabeli
df = (spark.createDataFrame(czujnik_temperatury, schema=schema)
      .withColumn("czas", to_timestamp("czas")))
df1 = (spark.createDataFrame(czujnik_temperatury, schema=SCHEMA)
      .withColumn("czas", to_timestamp("czas")))

In [11]:
df.printSchema()

root
 |-- temperatura: double (nullable = true)
 |-- czas: timestamp (nullable = true)



In [12]:
df1.printSchema()

root
 |-- temperatura: double (nullable = true)
 |-- czas: timestamp (nullable = true)



In [13]:
df1.show(3) # tak samo będzie dla df

+-----------+-------------------+
|temperatura|               czas|
+-----------+-------------------+
|       12.5|2019-01-02 12:00:00|
|       17.6|2019-01-02 12:00:20|
|       14.6|2019-01-02 12:00:30|
+-----------+-------------------+
only showing top 3 rows



In [14]:
df.createOrReplaceTempView("df")
# tworzymy widoki po to żeby udostępnić naszą tabelę, z zabezpieczeniem, ponieważ nie jest to oddzielny obiekt tylko referencja. 
# Po zepsuciu możemy go odtworzyć

In [15]:
spark.sql("select czas, temperatura from df where temperatura > 21").show(5)

+-------------------+-----------+
|               czas|temperatura|
+-------------------+-----------+
|2019-01-02 12:01:15|       22.9|
|2019-01-02 12:03:25|       25.8|
|2019-01-02 12:02:40|       27.1|
+-------------------+-----------+



In [16]:
# Thumbling window

import pyspark.sql.functions as F

df2 = df.groupBy(F.window("czas","30 seconds")).count()
df2.show(truncate=False)

+------------------------------------------+-----+
|window                                    |count|
+------------------------------------------+-----+
|{2019-01-02 12:00:00, 2019-01-02 12:00:30}|2    |
|{2019-01-02 12:00:30, 2019-01-02 12:01:00}|1    |
|{2019-01-02 12:01:00, 2019-01-02 12:01:30}|1    |
|{2019-01-02 12:01:30, 2019-01-02 12:02:00}|1    |
|{2019-01-02 12:03:00, 2019-01-02 12:03:30}|1    |
|{2019-01-02 12:02:30, 2019-01-02 12:03:00}|1    |
+------------------------------------------+-----+



In [ ]:
# strumień składa się z 3 części: źródło, przetworzenie i wyjście

In [17]:
# rate - źródło testowe 
df = spark.readStream.format("rate").option("rowsPerSecond", 1).load()

In [19]:
%%file streamrate.py
## uruchom przez spark-submit streamrate.py

# połączenie do sparka - trzeba zawsze wpisać
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

df = (spark.readStream # read straam a nie samo read jak do df
      .format("rate") # źródło
      .option("rowsPerSecond", 1)
      .load()
)


query = (df.writeStream 
    .format("console") 
    .outputMode("append") 
    .option("truncate", False) 
    .start()
) 

query.awaitTermination()  # wyzwalacz zakończenia dajemy żeby sam się nie zakończył

Overwriting streamrate.py


In [21]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, window
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()

from pyspark.sql.functions import col, expr

df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )

query = (stream.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

In [20]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()

from pyspark.sql.functions import col, expr

df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )

stream_filtered = stream.filter(col("temperatura") > 28)

query = (stream_filtered.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

Batch ID: 0
+----+-----------+
|czas|temperatura|
+----+-----------+
+----+-----------+

Batch ID: 1
+----+-----------+
|czas|temperatura|
+----+-----------+
+----+-----------+

Batch ID: 2
+----+-----------+
|czas|temperatura|
+----+-----------+
+----+-----------+

Batch ID: 3
+----+-----------+
|czas|temperatura|
+----+-----------+
+----+-----------+

Batch ID: 4
+----+-----------+
|czas|temperatura|
+----+-----------+
+----+-----------+

Batch ID: 5
+----+-----------+
|czas|temperatura|
+----+-----------+
+----+-----------+

Batch ID: 0
+----+-----------+
|czas|temperatura|
+----+-----------+
+----+-----------+

Batch ID: 1
+-----------------------+------------------+
|czas                   |temperatura       |
+-----------------------+------------------+
|2026-04-21 10:43:50.502|27.710112694364415|
+-----------------------+------------------+

Batch ID: 2
+-----------------------+-----------------+
|czas                   |temperatura      |
+-----------------------+--------------